In [5]:
import os
import sys

project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

from Model.Decision_Tree import (
    DecisionTree,
    make_bins, assign_bin,
    k_fold_split,
    accuracy_score, precision_recall_f1
)
from mysklearn.mypytable import MyPyTable



# Load Clean Dataset

In [9]:
table = MyPyTable().load_from_file("../output_data/tmdb_clean_step1.csv")

X = []
y = []

for row in table.data:
    X.append([
        row[0], row[1], row[2], row[3], row[4], row[5]
    ])
    y.append(row[6])

len(X), X[0], y[0]


(4803, [237000000, 150.437577, 162, 7.2, 11800, 2009], 'Action')

# Discretize Numeric Features into Bins

In [10]:
cols = list(zip(*X))

bins = [make_bins(col) for col in cols]

Xd = [
    [assign_bin(row[i], bins[i]) for i in range(len(row))]
    for row in X
]

Xd[0]


['bin1', 'bin0', 'bin1', 'bin2', 'bin2', 'bin2']

# 10-Fold Cross-Validation and Confusion Matrix

In [18]:
def confusion_matrix_multi(y_true, y_pred, labels):
    index = {lab: i for i, lab in enumerate(labels)}
    cm = [[0 for _ in labels] for _ in labels]
    for t, p in zip(y_true, y_pred):
        if t in index and p in index:
            cm[index[t]][index[p]] += 1
    return cm

def print_confusion_matrix(cm, labels):
    print("true\\pred\t" + "\t".join(labels))
    for i, lab in enumerate(labels):
        row = "\t".join(str(v) for v in cm[i])
        print(f"{lab}\t{row}")

folds = k_fold_split(Xd, y, k=10)

accuracies = []
precisions = []
recalls = []
f1s = []

y_true_all = []
y_pred_all = []

for X_train, y_train, X_test, y_test in folds:
    clf = DecisionTree()
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)

    y_true_all.extend(y_test)
    y_pred_all.extend(y_pred)

    acc = accuracy_score(y_test, y_pred)
    prec, rec, f1 = precision_recall_f1(y_test, y_pred, positive_label="Action")

    accuracies.append(acc)
    precisions.append(prec)
    recalls.append(rec)
    f1s.append(f1)

print("Decision Tree (10-fold CV)")
print("Accuracy:", sum(accuracies)/len(accuracies))
print("Precision:", sum(precisions)/len(precisions))
print("Recall:", sum(recalls)/len(recalls))
print("F1:", sum(f1s)/len(f1s))

labels = sorted(list(set(y)))   

cm = confusion_matrix_multi(y_true_all, y_pred_all, labels)

print("\nDecision Tree — Confusion Matrix")
print_confusion_matrix(cm, labels)


Decision Tree (10-fold CV)
Accuracy: 0.43410584886128367
Precision: 0.5442694511726486
Recall: 0.14638397632046257
F1: 0.22949974231019565

Decision Tree — Confusion Matrix
true\pred	Action	Comedy	Drama	Other
Action	171	565	391	25
Comedy	40	938	455	31
Drama	47	420	919	22
Other	57	424	234	57
